In [71]:
suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(scater))
suppressPackageStartupMessages(library(batchelor))
suppressPackageStartupMessages(library(argparse))

here::i_am("mapping/run/mnn/mapping_mnn.R")

# Load mapping functions
source(here::here("mapping/run/mnn/mapping_functions_extended.R"))

# Load default settings
source(here::here("settings.R"))
source(here::here("utils.R"))

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/01_Eomes_RNA/code



In [72]:
# I/O
io$path2atlas <- io$atlas.basedir
io$path2query <- io$basedir

# START TEST ##
args = list()
args$atlas_stages <- c("E8.5")
args$query_samples <- opts$samples
args$query_sce <- io$rna.sce
args$query_sce <- paste0(io$basedir,"/processed/SingleCellExperiment.rds")
args$atlas_sce <- io$rna.atlas.sce
args$query_metadata <- paste0(io$basedir,"/results/rna/doublet_detection/sample_metadata_after_doublets.txt.gz")
args$atlas_metadata <- io$rna.atlas.metadata
args$test <- TRUE
args$npcs <- 5
args$n_neighbours <- 25
args$use_marker_genes <- FALSE
args$cosine_normalisation <- FALSE
args$outdir <- paste0(io$basedir,"/results/rna/mapping/test")
# END TEST ##

In [73]:
opts$samples

[1] "SLX-20795_SITTH11_HKTG2DRXY" "SLX-20795_SITTH10_HKTG2DRXY"
[3] "SLX-20795_SITTG11_HKTG2DRXY" "SLX-20795_SITTG10_HKTG2DRXY"
[5] "SLX-20795_SITTF11_HKTG2DRXY" "SLX-20795_SITTB11_HKTG2DRXY"
[7] "SLX-20795_SITTA12_HKTG2DRXY" "SLX-20795_SITTA11_HKTG2DRXY"

In [74]:
if (isTRUE(args$test)) print("Test mode activated...")


[1] "Test mode activated..."


In [75]:
meta_query <- fread(args$query_metadata) %>% 
  .[pass_rnaQC==TRUE & doublet_call==FALSE]

In [76]:
################
## Load query ##
################

# Load cell metadata
meta_query <- fread(args$query_metadata) %>% 
  .[pass_rnaQC==TRUE & doublet_call==FALSE & sample%in%args$query_samples]
if (isTRUE(args$test)) meta_query <- head(meta_query,n=1000)

# Load SingleCellExperiment
sce_query <- load_SingleCellExperiment(args$query_sce, cells = meta_query$cell, remove_non_expressed_genes = TRUE)

# Update colData
tmp <- meta_query %>% .[cell%in%colnames(sce_query)] %>% setkey(cell) %>% .[colnames(sce_query)]
stopifnot(tmp$cell == colnames(sce_query))
colData(sce_query) <- tmp %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce_query),] %>% DataFrame()



In [77]:
sce_query

class: SingleCellExperiment 
dim: 13315 1000 
metadata(0):
assays(1): counts
rownames(13315): Xkr4 Sox17 ... DHRSX tomato-td
rowData names(0):
colnames(1000): SLX-20795_SITTA11_HKTG2DRXY#AAACCCAGTTCTCTAT-1
  SLX-20795_SITTA11_HKTG2DRXY#AAACGCTAGTGATTCC-1 ...
  SLX-20795_SITTA12_HKTG2DRXY#ACGATCAAGTATCCTG-1
  SLX-20795_SITTA12_HKTG2DRXY#ACGATCACAGCACAAG-1
colData names(13): sample barcode ... doublet_score doublet_call
reducedDimNames(0):
mainExpName: RNA
altExpNames(0):

In [78]:
################
## Load atlas ##
################

# Load cell metadata
meta_atlas <- fread(args$atlas_metadata) %>%
  .[stage%in%args$atlas_stages] %>%
  .[,sample:=factor(sample)]

# Filter
if (isTRUE(args$test)) meta_atlas <- head(meta_atlas,n=1000)

# Load SingleCellExperiment
sce_atlas <- load_SingleCellExperiment(args$atlas_sce, normalise = TRUE, cells = meta_atlas$cell, remove_non_expressed_genes = TRUE)

# Update colData
tmp <- meta_atlas %>% .[cell%in%colnames(sce_atlas)] %>% setkey(cell) %>% .[colnames(sce_atlas)]
stopifnot(tmp$cell == colnames(sce_atlas))
colData(sce_atlas) <- tmp %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce_atlas),] %>% DataFrame()

# Sanity cehcks
stopifnot(sum(is.na(rownames(sce_atlas)))==0)
stopifnot(sum(duplicated(rownames(sce_atlas)))==0)

In [79]:
#####################
## Define gene set ##
#####################

# Get gene metadata
gene_metadata <- fread(io$gene_metadata) %>% .[,c("chr","ens_id","symbol")] %>%
  .[symbol!="" & ens_id%in%rownames(sce_atlas)] %>%
  .[!duplicated(symbol)]

rownames(sce_atlas) = gene_metadata[match(rownames(sce_atlas), ens_id), symbol]

# Imprinted genes
imprint = gene_metadata[c(grep('maternally', gene_metadata$description),
                       grep('paternally', gene_metadata$description)), symbol]
#Other imprinted genes: 
#- Nnat (https://www.genecards.org/cgi-bin/carddisp.pl?gene=NNAT)
#- Grb10 (https://www.genecards.org/cgi-bin/carddisp.pl?gene=GRB10)

# Intersect genes
genes.intersect <- intersect(rownames(sce_query), rownames(sce_atlas))

# Filter some genes manually
genes.intersect <- genes.intersect[grep("^Rik|Rik$|^mt-|^Rps|^Rpl|^Gm",genes.intersect,invert=T)] # filter out non-informative genes
genes.intersect <- genes.intersect[grep("^Hbb|^Hba",genes.intersect,invert=T)] # test removing Haem genes 
genes.intersect <- genes.intersect[!genes.intersect %in% c(imprint, 'Grb10', 'Nnat')] # remove imprinted genes
genes.intersect <- genes.intersect[!genes.intersect %in% c("Xist", "Tsix")] # remove Xist & Tsix
genes.intersect <- genes.intersect[!genes.intersect=="tomato-td"] # remove tomato itself
genes.intersect <- genes.intersect[!genes.intersect %in% gene_metadata[chr=="chrY",symbol]] # no genes on y-chr 

# Subset SingleCellExperiment objects
sce_query  <- sce_query[genes.intersect,]
sce_atlas <- sce_atlas[genes.intersect,]

In [80]:
#######################
## Feature selection ##
#######################

if (args$use_marker_genes) {
  # Load marker genes
  marker_genes.dt <- fread(io$rna.atlas.marker_genes)
  genes_to_use <- genes.intersect[genes.intersect %in% unique(marker_genes.dt$gene)]
} else {  
  # Calculate mean-variance relationship and extract HVGs
  decomp <- modelGeneVar(sce_atlas, block=sce_atlas$sample)
  # genes_to_use <- rownames(decomp)[decomp$p.value<=0.01 & decomp$mean>0.1]
  genes_to_use <- decomp[order(decomp$FDR),] %>% head(n=2500) %>% rownames
}

stopifnot(genes_to_use%in%rownames(sce_atlas))
stopifnot(genes_to_use%in%rownames(sce_query))

sce_query = sce_query[genes_to_use,]
sce_atlas = sce_atlas[genes_to_use,]

In [81]:
head(genes_to_use, 20)

[1] "Tmsb4x"   "Tmsb10"   "Mest"     "Marcksl1" "Car2"     "Serpinh1"
 [7] "Tuba1a"   "Crabp1"   "Crabp2"   "Meg3"     "Myl7"     "Myl4"    
[13] "Acta2"    "Id3"      "Fth1"     "Actc1"    "Mdk"      "Krt18"   
[19] "H19"      "Cited4"

In [82]:
head(meta_atlas)

cell,barcode,sample,stage,sequencing.batch,doublet,stripped,celltype,umapX,umapY,nFeature_RNA,nCount_RNA,index,celltype_extended
<chr>,<chr>,<fct>,<chr>,<int>,<lgl>,<lgl>,<chr>,<dbl>,<dbl>,<int>,<int>,<dbl>,<chr>
cell_36865,AAACATACCACTGA,17,E8.5,2,FALSE,FALSE,Neural_crest,1.2261451,-7.987035,3850,18477,36865,Migratory_neural_crest
cell_36866,AAACATTGCAACTG,17,E8.5,2,FALSE,FALSE,Allantois,-6.2947984,9.852460,4223,18300,36866,Mesenchyme
cell_36867,AAACATTGGACGAG,17,E8.5,2,FALSE,FALSE,Erythroid3,10.3565961,6.400839,2307,11807,36867,Erythroid
cell_36869,AAACCGTGGCTAAC,17,E8.5,2,FALSE,FALSE,Forebrain_Midbrain_Hindbrain,0.9109553,-5.479232,3591,16119,36869,Dorsal_spinal_cord_progenitors
cell_36870,AAACCGTGTATCTC,17,E8.5,2,FALSE,FALSE,Somitic_mesoderm,1.5489777,1.385426,3623,14522,36870,Caudal_mesoderm
cell_36871,AAACGCACCCAACA,17,E8.5,2,FALSE,FALSE,Gut,-5.9116590,-10.271350,4018,18356,36871,Hindgut


In [88]:
source(here::here("mapping/run/mnn/mapping_functions_extended.R"))

mapping  <- mapWrap(
  sce_atlas = sce_atlas,
  meta_atlas = meta_atlas,
  sce_query = sce_query,
  meta_query = meta_query,
  genes = genes_to_use,
  npcs = args$npcs,
  k = args$n_neighbours,
  cosineNorm = args$cosine_normalisation,
  order = NULL
)

Normalizing joint dataset using cosineNorm=FALSE...

Done


2500 Genes provided...

Performing PCA...

Done


Batch effect correction for the atlas...

Done


MNN mapping...

Done


Computing mapping scores...

Done


Writing output...

Done




# MapWrap unpacked

In [48]:
  sce_atlas = sce_atlas
  meta_atlas = meta_atlas
  sce_query = sce_query
  meta_query = meta_query
  genes = genes_to_use
  npcs = args$npcs
  k = args$n_neighbours
  cosineNorm = args$cosine_normalisation
  order = NULL

In [49]:
   
  # Normalisation
  message(sprintf("Normalizing joint dataset using cosineNorm=%s...",cosineNorm))
  sce_all <- joint.normalisation(sce_query, sce_atlas, cosineNorm)
  message("Done\n")

Normalizing joint dataset using cosineNorm=FALSE...

Done




In [50]:
  
  # Feature selection
  if (is.null(genes)) {
    message("Genes not provided. Computing highly variable genes...")
    # hvgs <- getHVGs(sce_all, block=c(meta_atlas$sample, meta_query$sample))
    genes <- getHVGs(sce_all, block=sce_all$block)
    message("Done\n")
  } else {
    message(sprintf("%d Genes provided...",length(genes)))
  }

2500 Genes provided...



In [51]:
  # Dimensionality reduction
  message("Performing PCA...")
  big_pca <- multiBatchPCA(
    sce_all,
    batch = sce_all$block,
    subset.row = genes,
    d = npcs,
    preserve.single = TRUE,
    assay.type = if (cosineNorm) "cosineNorm" else "logcounts"
  )[[1]]
  rownames(big_pca) <- colnames(sce_all) 
  atlas_pca <- big_pca[1:ncol(sce_atlas),]
  query_pca   <- big_pca[-(1:ncol(sce_atlas)),]
  message("Done\n")

Performing PCA...

Done




In [52]:
  
  # Batch effect correction for the atlas
  message("Batch effect correction for the atlas...")  
  order_df        <- meta_atlas[!duplicated(meta_atlas$sample), c("stage", "sample")]
  order_df$ncells <- sapply(order_df$sample, function(x) sum(meta_atlas$sample == x))
  order_df$stage  <- factor(order_df$stage, levels = rev(c("E9.5",
                                       "E9.25",
                                       "E9.0",
                                       "E8.75",
                                       "E8.5",
                                       "E8.25",
                                       "E8.0",
                                       "E7.75",
                                       "E7.5",
                                       "E7.25",
                                       "mixed_gastrulation",
                                       "E7.0",
                                       "E6.75",
                                       "E6.5")))
  order_df       <- order_df[order(order_df$stage, order_df$ncells, decreasing = TRUE),]
  order_df$stage <- as.character(order_df$stage)
  
  set.seed(42)
  pca_atlas_corrected <- doBatchCorrect(counts         = logcounts(sce_atlas[genes,]), 
                                    timepoints      = meta_atlas$stage, 
                                    samples         = meta_atlas$sample, 
                                    timepoint_order = order_df$stage, 
                                    sample_order    = order_df$sample, 
                                    pc_override     = atlas_pca,
                                    npc             = npcs)
  message("Done\n")

Batch effect correction for the atlas...

Loading required package: BiocParallel

Done




In [54]:
  # Mapping query to batch-corrected atlas
  message("MNN mapping...")              
  # correct <- reducedMNN(rbind(pca_atlas_corrected, query_pca), batch = sce_all$block)[["corrected"]]
  correct <- reducedMNN(rbind(pca_atlas_corrected, query_pca),
                      # batch=c(rep("ATLAS", dim(meta_atlas)[1]), meta_query$sample),
                      batch = as.character(sce_all$block),
                      merge.order = order)$corrected
  pca_atlas_corrected <- correct[1:nrow(atlas_pca),]
  pca_query_corrected   <- correct[-(1:nrow(atlas_pca)),]

MNN mapping...



In [56]:
head(pca_query_corrected)
head(pca_atlas_corrected)

SLX-21143_SITTA2_HTJH3DSX2#AAACCCAAGATGTTCC-1,-14.987037,2.413490,24.6406411,-3.032402,5.767520
SLX-21143_SITTA2_HTJH3DSX2#AAACCCATCAGACCTA-1,-7.940671,7.905278,-14.2722171,-16.816031,10.581002
SLX-21143_SITTA2_HTJH3DSX2#AAACGAACAATGTTGC-1,-8.726419,8.758685,-0.9571918,-8.901286,-1.698947
SLX-21143_SITTA2_HTJH3DSX2#AAACGAACACACCAGC-1,-13.366618,3.451749,23.3918650,-5.954716,0.584606
SLX-21143_SITTA2_HTJH3DSX2#AAACGAAGTGATAGTA-1,-15.784547,1.576234,23.8164706,-1.795425,6.121257
SLX-21143_SITTA2_HTJH3DSX2#AAACGAAGTTAAGACA-1,-8.859923,9.763747,2.7037113,-5.096928,-4.358171


cell_361,-22.18608,-5.858202,-7.477659,13.22464068,-8.494032
cell_362,-12.44580,6.015637,2.600329,-1.15762130,2.156080
cell_363,-10.66507,8.345092,7.358897,-3.35566261,3.066436
cell_364,-10.77508,6.928303,9.191929,-4.99766121,-4.773238
cell_365,-12.98190,5.618987,1.879723,0.01229901,2.966236
cell_366,-10.58148,6.606713,6.264774,-6.22517346,-5.805861


In [59]:
meta_atlas$celltype = meta_atlas$celltype_extended_atlas

In [60]:
  mapping <- get_meta(pca_atlas = pca_atlas_corrected,
                      meta_atlas = meta_atlas,
                      pca_query = pca_query_corrected,
                      meta_query = meta_query,
                      k = k)
  message("Done\n")

Done




In [61]:

  

  


  



  
  # Mapping scores
  message("Computing mapping scores...") 
  out <- list()
  for (i in seq(from = 1, to = k)) {
    out$closest.cells[[i]]     <- sapply(mapping, function(x) x$cells.mapped[i])
    out$celltypes.mapped[[i]]  <- sapply(mapping, function(x) x$celltypes.mapped[i])
    out$cellstages.mapped[[i]] <- sapply(mapping, function(x) x$stages.mapped[i])
  }  
  multinomial.prob <- getMappingScore(out)
  message("Done\n")
  
  # Prepare output
  message("Writing output...") 
  out$pca_atlas_corrected <- pca_atlas_corrected
  out$pca_query_corrected <- pca_query_corrected
  ct <- sapply(mapping, function(x) x$celltype.mapped); is.na(ct) <- lengths(ct) == 0
  st <- sapply(mapping, function(x) x$stage.mapped); is.na(st) <- lengths(st) == 0
  cm <- sapply(mapping, function(x) x$cells.mapped[1]); is.na(cm) <- lengths(cm) == 0
  out$mapping <- data.frame(
      cell            = names(mapping), 
      celltype.mapped = unlist(ct),
      stage.mapped    = unlist(st),
      closest.cell    = unlist(cm))
  
  out$mapping <- cbind(out$mapping,multinomial.prob)
  out$pca <- big_pca
  message("Done\n")
  
  return(out)


Computing mapping scores...

Done


Writing output...

Done




In [62]:
str(out)

List of 7
 $ closest.cells      :List of 25
  ..$ : Named chr [1:1000] "cell_393" "cell_965" "cell_1108" "cell_393" ...
  .. ..- attr(*, "names")= chr [1:1000] "SLX-21143_SITTA2_HTJH3DSX2#AAACCCAAGATGTTCC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACCCATCAGACCTA-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACAATGTTGC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACACACCAGC-1" ...
  ..$ : Named chr [1:1000] "cell_592" "cell_711" "cell_1386" "cell_1434" ...
  .. ..- attr(*, "names")= chr [1:1000] "SLX-21143_SITTA2_HTJH3DSX2#AAACCCAAGATGTTCC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACCCATCAGACCTA-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACAATGTTGC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACACACCAGC-1" ...
  ..$ : Named chr [1:1000] "cell_398" "cell_1099" "cell_1196" "cell_2671" ...
  .. ..- attr(*, "names")= chr [1:1000] "SLX-21143_SITTA2_HTJH3DSX2#AAACCCAAGATGTTCC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACCCATCAGACCTA-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACAATGTTGC-1" "SLX-21143_SITTA2_HTJH3DSX2#AAACGAACACACCAGC-1" ...
  ..$ : Named chr